# Data processing Teams

In [1]:
import pandas as pd

DATA_PATH = "../../data/"

## Load data

In [2]:
def get_season(date):
    year = date.year
    if date.month >= 9:
        return f"{year}"
    else:
        return f"{year-1}"

In [3]:
team_stats = pd.read_csv(DATA_PATH + "/nba_database/TeamStatistics.csv")

# get the gametype (regular season or playoffs) from players
players = pd.read_csv(DATA_PATH + "/nba_database/PlayerStatistics.csv", dtype={10: str, 11: str, 15: str})
gametypes = players[["gameId", "gameType"]][players.gameType.isin(["Regular Season", "Playoffs"])].drop_duplicates()
team_stats = team_stats.merge(gametypes, on="gameId", how="inner")

# convert gameDateTimeEst to datetime and extract season
team_stats["gameDateTimeEst"] = pd.to_datetime(team_stats["gameDateTimeEst"])
team_stats["numMinutes"] = pd.to_numeric(team_stats["numMinutes"], errors="coerce")
team_stats["season"] = team_stats["gameDateTimeEst"].apply(get_season)

# keep only data from 1980
team_stats = team_stats[team_stats["gameDateTimeEst"] >= pd.Timestamp("1980-10-01")]

# rename reboundsTotal to rebounds
team_stats = team_stats.rename(columns={"reboundsTotal": "rebounds"}) 

display(team_stats)

,gameId,gameDateTimeEst,teamCity,teamName,teamId,opponentTeamCity,opponentTeamName,opponentTeamId,home,win,...,pointsFromTurnovers,pointsInThePaint,pointsSecondChance,timesTied,timeoutsRemaining,seasonWins,seasonLosses,coachId,gameType,season
0,22500960,2026-03-12 22:30:00,Chicago,Bulls,1610612741,Los Angeles,Lakers,1610612747,0,0.0,...,18.0,66.0,16.0,5.0,0.0,27.0,39.0,NaN,Regular Season,2025
1,22500960,2026-03-12 22:30:00,Los Angeles,Lakers,1610612747,Chicago,Bulls,1610612741,1,1.0,...,17.0,72.0,30.0,5.0,1.0,41.0,25.0,NaN,Regular Season,2025
2,22500959,2026-03-12 21:30:00,Boston,Celtics,1610612738,Oklahoma City,Thunder,1610612760,0,0.0,...,14.0,34.0,23.0,14.0,0.0,43.0,23.0,NaN,Regular Season,2025
3,22500959,2026-03-12 21:30:00,Oklahoma City,Thunder,1610612760,Boston,Celtics,1610612738,1,1.0,...,16.0,44.0,13.0,14.0,0.0,52.0,15.0,NaN,Regular Season,2025
4,22500958,2026-03-12 21:00:00,Denver,Nuggets,1610612743,San Antonio,Spurs,1610612759,0,1.0,...,7.0,56.0,15.0,0.0,0.0,41.0,26.0,NaN,Regular Season,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110127,28000002,1980-10-10 20:00:00,San Antonio,Spurs,1610612759,Denver,Nuggets,1610612743,0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Regular Season,1980
110128,28000006,1980-10-10 20:00:00,Seattle,SuperSonics,1610612760,Los Angeles,Lakers,1610612747,1,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Regular Season,1980
110129,28000008,1980-10-10 20:00:00,Utah,Jazz,1610612762,Portland,Trail Blazers,1610612757,1,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Regular Season,1980
110130,28000005,1980-10-10 20:00:00,Washington,Bullets,1610612764,Detroit,Pistons,1610612765,0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Regular Season,1980


In [4]:
# add salary data
salaries = pd.read_csv(DATA_PATH + "/nba_salary/team_salaries.csv")
salaries["season"] = salaries["season"].apply(lambda x: x.split("-")[0])
display(salaries)

,team,salary_usd,season
0,Cavaliers,14403000,1990
1,Knicks,13290000,1990
2,Pistons,12910000,1990
3,Lakers,12120000,1990
4,Hawks,11761000,1990
...,...,...,...
1051,Bucks,184516265,2025
1052,Bulls,184248613,2025
1053,Jazz,175671224,2025
1054,Grizzlies,160873801,2025


### Per game 

In [5]:
columns_to_exclude = ["q1Points", "q2Points", "q3Points", "q4Points", 
                      "benchPoints", "biggestLead", "biggestScoringRun", 
                      "leadChanges", "pointsFastBreak", "pointsFromTurnovers", 
                      "pointsInThePaint", "pointsSecondChance", "timesTied", 
                      "timeoutsRemaining", "seasonWins", "seasonLosses", "coachId"]

team_stats.drop(columns=columns_to_exclude).to_csv(DATA_PATH + "/processed/team_games.csv", index=False)

### Per season (for bubble & spider charts)

In [6]:
# per season stats:
# Points
# Rebounds
# Assists
# Steal 
# Block
# Turnover
# Shot accuracy percentage (3p%, 2p%, FT%) total
# Proportion of 3pts % 
# Number of shot attempted
# Number of shot made
# Fouls
# +/-
# Salary
# Nb wins
# Nb losses

columns_to_keep = ["season", "gameType", "teamId", "teamName", "teamCity", "teamScore", 
                   "opponentScore", "assists", "rebounds", "blocks", "steals", "turnovers", 
                   "fieldGoalsAttempted", "fieldGoalsMade", "threePointersAttempted", "threePointersMade", 
                   "freeThrowsAttempted", "freeThrowsMade", "foulsPersonal", "plusMinusPoints", "numMinutes", "gameId", "win"]

agg_rules = {
    "teamScore": "mean",
    "opponentScore": "mean",
    "assists": "mean",
    "rebounds": "mean",
    "blocks": "mean",
    "steals": "mean",
    "turnovers": "mean",
    "fieldGoalsMade": "sum",
    "fieldGoalsAttempted": "sum",
    "threePointersMade": "sum",
    "threePointersAttempted": "sum",
    "freeThrowsMade": "sum",
    "freeThrowsAttempted": "sum",
    "plusMinusPoints": "mean",
    "foulsPersonal": "mean",
    "numMinutes": "mean",
    "win": "sum",
    "gameId": "count"
}

group_cols = ["season", "teamId", "gameType", "teamName", "teamCity"]
grouped = team_stats[columns_to_keep].groupby(group_cols)

# season agregated stats per team
team_stats_season = grouped.agg(agg_rules)
team_stats_season = team_stats_season.rename(columns={"gameId": "gamesPlayed"})

# add season totals for main box-score stats
main_stats = ["teamScore", "opponentScore", "assists", "rebounds", "blocks", "steals", "turnovers"]
main_totals = grouped[main_stats].sum().rename(columns={col: f"{col}Total" for col in main_stats})
team_stats_season = team_stats_season.join(main_totals)

# use minimum game thresholds for regular season / playoffs
regular_season_min_games = 15
playoffs_min_games = 4

game_type_idx = team_stats_season.index.get_level_values("gameType")
team_stats_season = team_stats_season[
    ((game_type_idx == "Regular Season") & (team_stats_season["gamesPlayed"] >= regular_season_min_games)) |
    ((game_type_idx == "Playoffs") & (team_stats_season["gamesPlayed"] >= playoffs_min_games))
]

# calculate shooting percentages on season totals
team_stats_season["fieldGoalsPercentage"] = team_stats_season["fieldGoalsMade"] / team_stats_season["fieldGoalsAttempted"]
team_stats_season["threePointersPercentage"] = team_stats_season["threePointersMade"] / team_stats_season["threePointersAttempted"]
team_stats_season["freeThrowsPercentage"] = team_stats_season["freeThrowsMade"] / team_stats_season["freeThrowsAttempted"]
team_stats_season["proportionThreePoint"] = team_stats_season["threePointersAttempted"] / team_stats_season["fieldGoalsAttempted"]

team_stats_season["losses"] = team_stats_season["gamesPlayed"] - team_stats_season["win"]

team_stats_season_flat = team_stats_season.reset_index().copy()

# join salaries on personId and season
salaries = salaries.rename(columns={"salary_usd": "salary"})
team_stats_season_flat = team_stats_season_flat.merge(
    salaries,
    left_on=["teamName", "season"],
    right_on=["team", "season"],
    how="left"
)
team_stats_season_flat = team_stats_season_flat.drop(columns=["team"])


team_stats_season_flat.to_csv(DATA_PATH + "/processed/team_seasons.csv", index=False)
display(team_stats_season_flat)

,season,teamId,gameType,teamName,teamCity,teamScore,opponentScore,assists,rebounds,blocks,...,reboundsTotal,blocksTotal,stealsTotal,turnoversTotal,fieldGoalsPercentage,threePointersPercentage,freeThrowsPercentage,proportionThreePoint,losses,salary
0,1980,1610612737,Regular Season,Hawks,Atlanta,104.926829,108.024390,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,inf,inf,0.776534,NaN,51.0,NaN
1,1980,1610612738,Playoffs,Celtics,Boston,103.352941,97.294118,22.833333,47.333333,5.500000,...,284.0,33.0,40.0,101.0,1.367188,0.588235,0.757642,0.033203,5.0,NaN
2,1980,1610612738,Regular Season,Celtics,Boston,109.853659,103.975610,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,inf,inf,0.752429,NaN,20.0,NaN
3,1980,1610612739,Regular Season,Cavaliers,Cleveland,105.731707,110.585366,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,inf,inf,0.779119,NaN,54.0,NaN
4,1980,1610612741,Playoffs,Bulls,Chicago,103.500000,107.166667,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,inf,inf,0.762500,NaN,4.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1937,2025,1610612762,Regular Season,Jazz,Utah,117.409091,125.045455,29.484848,43.969697,3.636364,...,2902.0,240.0,560.0,997.0,0.464933,0.347296,0.800118,0.412584,46.0,175671224.0
1938,2025,1610612763,Regular Season,Grizzlies,Memphis,115.800000,118.307692,28.676923,43.723077,5.138462,...,2842.0,334.0,564.0,957.0,0.461082,0.355538,0.791968,0.422588,42.0,160873801.0
1939,2025,1610612764,Regular Season,Wizards,Washington,112.846154,123.876923,25.092308,42.953846,5.892308,...,2792.0,383.0,508.0,991.0,0.459891,0.354757,0.769886,0.401937,49.0,189949625.0
1940,2025,1610612765,Regular Season,Pistons,Detroit,117.230769,109.615385,26.846154,45.784615,6.415385,...,2976.0,417.0,683.0,918.0,0.478692,0.348166,0.759977,0.349991,18.0,188270076.0


## Metadata

In [7]:
metadata = pd.read_csv(DATA_PATH + "/nba_database/TeamHistories.csv")
colors = pd.read_csv(DATA_PATH + "/nba_team_colors/nba_team_colors.csv")

metadata = metadata[(metadata.league == "NBA") & (metadata.seasonActiveTill >= 1979) & (~metadata.teamCity.isin(["All-Star", "West", "East"]))]
metadata["teamSlug"] = metadata["teamCity"].str.lower().str.replace(" ", "-") + "-" + metadata["teamName"].str.lower().str.replace(" ", "-")
metadata["name"] = metadata["teamCity"] + " " + metadata["teamName"]
metadata["teamAbbrev"] = metadata["teamAbbrev"].str.strip()

metadata = metadata.merge(colors, left_on="name", right_on="Team", how="left")
metadata = metadata[["teamId", "teamAbbrev", "teamSlug", "Color1", "Color2", "Color3", "Color4", "Color5", "seasonFounded", "seasonActiveTill"]]

metadata.to_csv(DATA_PATH + "/processed/team_metadata.csv", index=False)
display(metadata.reset_index(drop=True))

,teamId,teamAbbrev,teamSlug,Color1,Color2,Color3,Color4,Color5,seasonFounded,seasonActiveTill
0,1610612737,ATL,atlanta-hawks,#C8102E,#FFC72C,#FFFFFF,#010101,#DBE2E9,1968,2100
1,1610612739,CLE,cleveland-cavaliers,#6F263D,#FFFFFF,#010101,#373A36,NaN,1970,2100
2,1610612740,NOH,new-orleans-hornets,#00778B,#280071,#FFC72C,#FFFFFF,#010101,2002,2004
3,1610612740,NOK,oklahoma-city-hornets,#00778B,#280071,#FFC72C,#FFFFFF,NaN,2005,2006
4,1610612740,NOH,new-orleans-hornets,#00778B,#280071,#FFC72C,#FFFFFF,#010101,2007,2012
5,1610612740,NOP,new-orleans-pelicans,#0C2340,#B9975B,#C8102E,#FFFFFF,NaN,2013,2100
6,1610612741,CHI,chicago-bulls,#BA0C2F,#010101,#FFFFFF,NaN,NaN,1966,2100
7,1610612742,DAL,dallas-mavericks,#0050B5,#0C2340,#9EA2A2,#FFFFFF,NaN,1980,2100
8,1610612743,DEN,denver-nuggets,#0C2340,#862633,#FFC72C,#FFFFFF,NaN,1976,2100
9,1610612744,GSW,golden-state-warriors,#1D4289,#FFC72C,#FFFFFF,#010101,NaN,1971,2100
